# Phase 3 — embedding precompute (Colab GPU)

Runs `src/data.py` then `src/embed.py` to encode every dataset sentence once
with the frozen `all-mpnet-base-v2` backbone, writing the cached `.npz` files to
a **Google Drive** folder (Colab storage is ephemeral).

Runtime: set **Runtime → Change runtime type → GPU** (T4 is fine). Full run is
~15–30 min on a T4. `embed.py` skips any split already cached, so a disconnect
just means re-running the last cell.

Everything after this (features, heads, pilot, full grid, analysis) runs on CPU
off the cached `.npz` files — no GPU needed again.

In [ ]:
!git clone https://github.com/ryanteachman/sbert-head-ablation.git
%cd sbert-head-ablation

In [ ]:
# Colab ships a CUDA torch; install only what data.py / embed.py additionally need.
!pip install -q "datasets==4.0.0" hf_xet "sentence-transformers==5.1.2" pyarrow pyyaml scikit-learn
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EMBED_DIR = '/content/drive/MyDrive/sbert-head-ablation/embeddings'
import os; os.makedirs(EMBED_DIR, exist_ok=True)
print('embeddings ->', EMBED_DIR)

## 1. Build processed splits
Deterministic (fixed QQP split seed) — identical to a local run. Verifies every
split size against `PROTOCOL.md` §5.

In [ ]:
!python src/data.py

## 2. Sanity-check the encoder, then encode
`--verify` confirms L2-normalized + deterministic output before the real run.

In [ ]:
!python src/embed.py --verify

In [ ]:
!python src/embed.py --embed-dir "{EMBED_DIR}"    # add --force to re-encode

## 3. Inspect the cache

In [ ]:
import json, numpy as np, pathlib
print(json.dumps(json.load(open(f'{EMBED_DIR}/meta.json')), indent=2))
for p in sorted(pathlib.Path(EMBED_DIR).rglob('*.npz')):
    z = np.load(p)
    print(f'{p.relative_to(EMBED_DIR)}  uniq_emb{z["uniq_emb"].shape} {z["uniq_emb"].dtype}  '
          f'pairs={len(z["label"]):,}  {p.stat().st_size/1e6:.0f} MB')

## Done
The `.npz` files are in Drive at `MyDrive/sbert-head-ablation/embeddings/`.
For local downstream work, download that folder once (~2–3 GB), or run the
remaining phases here in Colab against `--embed-dir` / the same path.